# Drug–gene network analysis of adverse drug reaction (ADR) predictability

Pipeline:

1. Load the drug–side effect and drug–gene binary matrices
2. Jaccard similarity between drugs (gene profiles, side-effect profiles)
3. Build the drug similarity network (gene similarity threshold)
4. Harmonic-function label propagation with 5-fold cross-validation, plus a k-NN baseline
5. Rank ADRs by mean AUROC
6. Export gene lists of the top and bottom ADRs for enrichment analysis

## 1. Setup

In [41]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from scipy.stats import spearmanr, mannwhitneyu, hypergeom
from sklearn.model_selection import KFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)

In [42]:
# ---------------- configuration ----------------
DATA_DIR  = "/content/drive/MyDrive"
SE_FILE   = os.path.join(DATA_DIR, "drug_sideeffect_binary_matrix.csv")   # drugs x side effects (0/1)
GENE_FILE = os.path.join(DATA_DIR, "intersection_DGIdb_mat.tsv")          # drugs x genes (0/1)

SIM_THRESHOLD      = 0.3     # a drug is kept if it has at least one neighbour with gene similarity >= this
N_SPLITS           = 5       # cross-validation folds (over drugs)
SEED               = 27
EPS                = 1e-6    # regularisation added to L_uu
DECISION_THRESHOLD = 0.5     # cut-off used for precision / recall / F1 / specificity / accuracy
KNN_K_VALUES       = [5, 10, 15, 20]

# Harmonic scores that are equal in theory can differ in the last decimal places, and AUROC/AUPRC
# then rank them in an arbitrary order. Set to e.g. 10 to round the scores so that such ties count as ties.
# Without rounding, the ranking of ADRs with nearly equal AUROC changes from one computer to another. None = no rounding.
TIE_DECIMALS       = 10

# The gene file contains a few entries equal to 2. True = treat every non-zero entry as an interaction (0/1 matrix,
# proper Jaccard similarity).
BINARISE_INPUT = True

# If True the harmonic function only uses edges >= SIM_THRESHOLD.
# False = all pairwise similarities between the retained drugs.
USE_THRESHOLDED_EDGES = False

# ADR ranking
N_EXTREME          = 10      # size of the top and bottom groups
MIN_POSITIVE_DRUGS = 10      # an ADR is ranked only if at least this many drugs have it AND at least this many do not
REQUIRE_ALL_FOLDS  = True    # ... and only if it could be evaluated in every fold
RANK_BY            = "AUROC_mean"   # does not depend on how common the ADR is; alternatives: "AUPRC_gain_mean", "AUPRC_mean"
FDR_ALPHA          = 0.05    # Benjamini-Hochberg level used throughout

# Extra analyses (each adds run time)
RUN_NETWORK_CONTROL       = True                         # shuffled-network negative control
RUN_THRESHOLD_SENSITIVITY = True                         # repeat the cross-validation for other thresholds
THRESHOLDS                = [0.1, 0.2, 0.3, 0.4, 0.5]
RUN_ENRICHR               = True                         # KEGG / GO enrichment through the Enrichr web service
ENRICHR_LIBRARIES         = ["KEGG", "GO_Biological_Process"]   # newest version of each

OUT_DIR = "outputs"
FIG_DIR = "figures"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

In [43]:
# Mount Google Drive when running on Colab; skipped elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running on Colab - reading files from", DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Data

In [44]:
def load_binary_matrix(path, sep, index_col=None):
    """Read a drug x feature 0/1 matrix with drug names as the index."""
    df = pd.read_csv(path, sep=sep)
    if index_col is not None:
        df = df.set_index(index_col)
    elif isinstance(df.index, pd.RangeIndex):       # names are in the first column
        df = df.set_index(df.columns[0])
    df = df.fillna(0).astype(int)
    df.index = df.index.astype(str)
    df.index.name = None
    df.columns.name = None
    if df.index.has_duplicates:
        raise ValueError(f"Duplicated drug names in {path}")
    n_other = int((~np.isin(df.to_numpy(), [0, 1])).sum())
    if n_other:
        print(f"Note: {os.path.basename(path)} has {n_other} entries that are not 0/1 "
              f"(largest value {df.to_numpy().max()}).", "They are set to 1." if BINARISE_INPUT else "They are kept as they are.")
        if BINARISE_INPUT:
            df = (df > 0).astype(int)
    return df


drugs_se  = load_binary_matrix(SE_FILE, sep=",", index_col="drug_name")
drug_gene = load_binary_matrix(GENE_FILE, sep="\t")

print(f"Side-effect matrix: {drugs_se.shape[0]} drugs x {drugs_se.shape[1]} side effects")
print(f"Gene matrix:        {drug_gene.shape[0]} drugs x {drug_gene.shape[1]} genes")

Note: intersection_DGIdb_mat.tsv has 6 entries that are not 0/1 (largest value 2). They are set to 1.
Side-effect matrix: 1020 drugs x 5599 side effects
Gene matrix:        777 drugs x 2022 genes


## 3. Drug similarity and network

In [45]:
def jaccard_similarity_matrix(binary_df):
    """Pairwise Jaccard similarity between the rows of a 0/1 DataFrame."""
    A = binary_df.to_numpy(dtype=float)
    inter = A @ A.T
    row_sum = A.sum(axis=1)
    union = row_sum[:, None] + row_sum[None, :] - inter
    sim = np.divide(inter, union, out=np.zeros_like(inter), where=union != 0)
    np.fill_diagonal(sim, 1.0)
    return pd.DataFrame(sim, index=binary_df.index, columns=binary_df.index)


# drugs present in both datasets (order of the gene file is kept, the CV split depends on it)
shared_drugs = drug_gene.index.intersection(drugs_se.index, sort=False)
print("Drugs in both datasets:", len(shared_drugs))

druggene_jaccard = jaccard_similarity_matrix(drug_gene)                    # drug-drug similarity from gene profiles
drugse_jaccard = jaccard_similarity_matrix(drugs_se)                       # drug-drug similarity from side-effect profiles

dg_sim = druggene_jaccard.loc[shared_drugs, shared_drugs]
se_sim = drugse_jaccard.loc[shared_drugs, shared_drugs]
S = dg_sim.to_numpy().copy()
np.fill_diagonal(S, 0.0)                                                   # no self-similarity

# keep drugs with at least one neighbour at or above the threshold
keep = (S >= SIM_THRESHOLD).any(axis=1)
common_drugs = shared_drugs[keep].tolist()
n_drugs = len(common_drugs)

drug_gene_matrix = drug_gene.loc[common_drugs]            # drugs x genes, analysed drugs only
drug_se_matrix   = drugs_se.loc[common_drugs]             # drugs x side effects, analysed drugs only

W_full = S[np.ix_(keep, keep)]                            # all pairwise gene similarities
W_thr  = np.where(W_full >= SIM_THRESHOLD, W_full, 0.0)   # edges >= threshold only
W = W_thr if USE_THRESHOLDED_EDGES else W_full            # matrix used for propagation and k-NN

drug_gene_sim = druggene_jaccard.loc[common_drugs, common_drugs]   # gene similarity of the analysed drugs
se_sim = drugse_jaccard.loc[common_drugs, common_drugs]            # side-effect similarity of the analysed drugs

print(f"Drugs retained at threshold {SIM_THRESHOLD}: {n_drugs} "
      f"({100 * n_drugs / len(shared_drugs):.0f}% of {len(shared_drugs)})")
print("Side-effect matrix used for prediction:", drug_se_matrix.shape)

Drugs in both datasets: 777
Drugs retained at threshold 0.3: 536 (69% of 777)
Side-effect matrix used for prediction: (536, 5599)


In [46]:
# Network of the retained drugs (edges = gene similarity >= threshold)
threshold_graph = nx.Graph()
threshold_graph.add_nodes_from(common_drugs)
rows_i, cols_j = np.nonzero(np.triu(W_thr, k=1))
for i, j in zip(rows_i, cols_j):
    threshold_graph.add_edge(common_drugs[i], common_drugs[j],
                             drug_gene_similarity=W_thr[i, j],
                             drug_se_similarity=se_sim.iat[i, j])

full_graph = nx.from_numpy_array((W_full > 0).astype(int))

network_summary = pd.Series({
    "nodes": threshold_graph.number_of_nodes(),
    "edges (similarity >= threshold)": threshold_graph.number_of_edges(),
    "connected components (edges >= threshold)": nx.number_connected_components(threshold_graph),
    "largest component (edges >= threshold)": len(max(nx.connected_components(threshold_graph), key=len)),
    "edges (any similarity > 0)": full_graph.number_of_edges(),
    "connected components (any similarity > 0)": nx.number_connected_components(full_graph),
})
print(network_summary.to_string())
print("\nMatrix used for propagation:", "thresholded edges" if USE_THRESHOLDED_EDGES else "all pairwise similarities")

nodes                                          536
edges (similarity >= threshold)               2416
connected components (edges >= threshold)       41
largest component (edges >= threshold)         407
edges (any similarity > 0)                   36726
connected components (any similarity > 0)        3

Matrix used for propagation: all pairwise similarities


In [47]:
# Helpers to look things up by name
def drugs_for_adr(adr):
    """Analysed drugs annotated with a given side effect."""
    return drug_se_matrix.index[drug_se_matrix[adr] == 1].tolist()


def genes_for_drugs(drugs):
    """Number of the given drugs that interact with each gene (most shared first)."""
    counts = (drug_gene_matrix.loc[drugs] > 0).sum(axis=0)
    return counts[counts > 0].sort_values(ascending=False)


def neighbours(drug):
    """Network neighbours of a drug with both similarity scores."""
    rows = [{"similar_drug": nb, **attrs} for nb, attrs in threshold_graph[drug].items()]
    return pd.DataFrame(rows).sort_values("drug_gene_similarity", ascending=False).reset_index(drop=True)


neighbours(common_drugs[0])

,similar_drug,drug_gene_similarity,drug_se_similarity
0,MEMANTINE,0.333333,0.315068


In [48]:
# Per-drug table: neighbours above the threshold with both similarity scores, genes and side effects
rows = []
for i, drug in enumerate(common_drugs):
    nb = np.nonzero(W_thr[i])[0]
    rows.append({
        "drug_name": drug,
        "similar_drugs": [common_drugs[j] for j in nb],
        "drug_gene_similarity_scores": W_thr[i, nb].tolist(),
        "drug_se_similarity_scores": se_sim.to_numpy()[i, nb].tolist(),
        "associated_genes": drug_gene_matrix.columns[drug_gene_matrix.loc[drug] > 0].tolist(),
        "associated_side_effects": drug_se_matrix.columns[drug_se_matrix.loc[drug] == 1].tolist(),
    })
common_df = pd.DataFrame(rows)
print(common_df.shape)
common_df.head()

(536, 6)


,drug_name,similar_drugs,drug_gene_similarity_scores,drug_se_similarity_scores,associated_genes,associated_side_effects
0,ACAMPROSATE,[MEMANTINE],[0.3333333333333333],[0.3150684931506849],"[GRIN2B, GRIN3A, GRM5]","[ABDOMINAL DISTENSION, ABDOMINAL PAIN, ABNORMA..."
1,ACARBOSE,[MIGLITOL],[0.42857142857142855],[0.4772727272727273],"[AMY2A, GAA, LPL, MGA, MGAM, SI, SLC2A4]","[ABDOMINAL PAIN, ANGIOPATHY, CONSTIPATION, DER..."
2,ACEBUTOLOL,"[BETAXOLOL, BISOPROLOL, BRETYLIUM, CARTEOLOL, ...","[0.3333333333333333, 0.3333333333333333, 0.5, ...","[0.3187919463087248, 0.3779527559055118, 0.176...","[ADRB1, TG]","[ABDOMINAL PAIN, ABNORMAL DREAMS, ABNORMAL VIS..."
3,ACETAZOLAMIDE,"[BRINZOLAMIDE, METHAZOLAMIDE]","[0.38461538461538464, 0.375]","[0.09917355371900827, 0.5833333333333334]","[AQP1, AQP4, CA1, CA12, CA14, CA2, CA3, CA4, C...","[AGITATION, AGRANULOCYTOSIS, ANAPHYLACTIC SHOC..."
4,ACETYLCHOLINE,"[ACLIDINIUM, ATROPINE, BETHANECHOL, CARBACHOL,...","[0.8333333333333334, 0.38461538461538464, 0.83...","[0.0, 0.02843601895734597, 0.10526315789473684...","[ACE, CHRM1, CHRM2, CHRM3, CHRM4, CHRM5]","[BRADYCARDIA, CORNEAL DECOMPENSATION, CORNEAL ..."


## 4. Harmonic-function prediction and k-NN baseline (5-fold cross-validation)

In [49]:
def harmonic_system(W, train_idx, test_idx, eps=EPS):
    """
    Blocks of the graph Laplacian for one train/test split.
    The harmonic solution for a label vector y is  f_u = solve(A, B @ y[train_idx])
    with A = L_uu + eps*I and B = -L_ul.
    """
    W = np.array(W, dtype=float)
    np.fill_diagonal(W, 0.0)
    L = np.diag(W.sum(axis=1)) - W                      # graph Laplacian L = D - W
    A = L[np.ix_(test_idx, test_idx)] + eps * np.eye(len(test_idx))
    B = -L[np.ix_(test_idx, train_idx)]
    return A, B


def harmonic_prediction(W, y, train_idx, eps=EPS):
    """Scores for all drugs for one side effect; training drugs keep their labels."""
    y = np.asarray(y, dtype=float)
    test_idx = np.setdiff1d(np.arange(len(y)), train_idx)
    A, B = harmonic_system(W, train_idx, test_idx, eps)
    f = y.copy()
    f[test_idx] = np.linalg.solve(A, B @ y[train_idx])
    return f


def knn_scores(W, Y, train_idx, test_idx, k):
    """Fraction of the k most similar training drugs that have each side effect."""
    sims = W[np.ix_(test_idx, train_idx)]
    nn = np.argsort(-sims, axis=1, kind="stable")[:, :k]     # stable: equal similarities keep drug order
    return Y[train_idx][nn].mean(axis=1)                # (n_test, n_side_effects)


def minmax_scale(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-12)


roc_grid = np.linspace(0, 1, 200)
pr_grid = np.linspace(0, 1, 200)


def evaluate(Y_train, Y_test, score_fn, side_effects, scale_scores, curves=False):
    """
    Per-side-effect test metrics for one fold. score_fn(j) returns the test scores of side effect j.
    With curves=True it also returns the mean ROC and PR curves of the fold.
    """
    rows, tprs, precs = [], [], []
    for j, name in enumerate(side_effects):
        y_true = Y_test[:, j]
        n_pos_test, n_pos_train = int(y_true.sum()), int(Y_train[:, j].sum())
        # a side effect needs positives in both sets and at least one negative test drug
        if n_pos_train == 0 or n_pos_test == 0 or n_pos_test == len(y_true):
            continue
        score = score_fn(j)
        if TIE_DECIMALS is not None:
            score = np.round(score, TIE_DECIMALS)
        y_pred = (minmax_scale(score) if scale_scores else score) >= DECISION_THRESHOLD

        tp = int((y_pred & (y_true == 1)).sum())
        fp = int((y_pred & (y_true == 0)).sum())
        fn = n_pos_test - tp
        tn = len(y_true) - n_pos_test - fp
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn)
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        rows.append({
            "side_effect": name,
            "AUC_ROC": roc_auc_score(y_true, score),
            "AUPRC": average_precision_score(y_true, score),
            "Precision": precision, "Recall": recall, "F1_Score": f1,
            "Specificity": tn / (tn + fp), "Accuracy": (tp + tn) / len(y_true),
            "TP": tp, "TN": tn, "FP": fp, "FN": fn,
            "n_pos_train": n_pos_train, "n_pos_test": n_pos_test,
            "prevalence_test": n_pos_test / len(y_true),         # AUPRC of a random ranking
        })
        if curves:
            fpr, tpr, _ = roc_curve(y_true, score)
            prec, rec, _ = precision_recall_curve(y_true, score)
            tprs.append(np.interp(roc_grid, fpr, tpr))
            precs.append(np.interp(pr_grid, rec[::-1], prec[::-1]))
    if curves:
        return rows, np.mean(tprs, axis=0), np.mean(precs, axis=0)
    return rows

In [ ]:
METRICS = ["AUC_ROC", "AUPRC", "Precision", "Recall", "F1_Score", "Specificity", "Accuracy"]

Y_all = drug_se_matrix.to_numpy(dtype=float)
side_effects = drug_se_matrix.columns.tolist()

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(kf.split(common_drugs))                     # one split, reused everywhere below

harmonic_rows, knn_rows = [], []
fold_curves = {}                                         # mean ROC / PR curve of each fold (harmonic)
oof_scores = np.zeros_like(Y_all)                        # harmonic score of every drug from the fold in which it was a test drug

for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    print(f"Fold {fold}: {len(train_idx)} training drugs, {len(test_idx)} test drugs")
    Y_train, Y_test = Y_all[train_idx], Y_all[test_idx]

    A, B = harmonic_system(W, train_idx, test_idx)
    rows, mean_tpr, mean_prec = evaluate(Y_train, Y_test,
                                         lambda j: np.linalg.solve(A, B @ Y_train[:, j]),
                                         side_effects, scale_scores=True, curves=True)
    harmonic_rows += [{"fold": fold, **r} for r in rows]
    fold_curves[fold] = {"tpr": mean_tpr, "prec": mean_prec}
    oof_scores[test_idx] = np.linalg.solve(A, B @ Y_train)

    for k in KNN_K_VALUES:
        S_knn = knn_scores(W, Y_all, train_idx, test_idx, k)
        rows = evaluate(Y_train, Y_test, lambda j: S_knn[:, j], side_effects, scale_scores=False)
        knn_rows += [{"k": k, "fold": fold, **r} for r in rows]

if TIE_DECIMALS is not None:
    oof_scores = np.round(oof_scores, TIE_DECIMALS)
results_df = pd.DataFrame(harmonic_rows)
baseline_all_df = pd.DataFrame(knn_rows)
print("\nEvaluated side effect x fold combinations:", len(results_df))

Fold 1: 428 training drugs, 108 test drugs
Fold 2: 429 training drugs, 107 test drugs
Fold 3: 429 training drugs, 107 test drugs
Fold 4: 429 training drugs, 107 test drugs


## 5. Performance summary

In [ ]:
fold_summary = results_df.groupby("fold")[METRICS].mean()
overall_summary = results_df[METRICS].mean()

print("Harmonic propagation, mean per fold:")
display(fold_summary.round(3))
print("Range across folds:")
display(fold_summary.agg(["min", "max"]).round(3))

In [ ]:
# k-NN baseline for every k, and the comparison with harmonic propagation
knn_by_k = baseline_all_df.groupby("k")[METRICS].mean()
print("k-NN baseline, mean over all folds:")
display(knn_by_k.round(3))

best_k = knn_by_k["AUPRC"].idxmax()
baseline_results_df = baseline_all_df[baseline_all_df["k"] == best_k]        # the reported baseline
baseline_summary = knn_by_k.loc[best_k]

comparison = pd.DataFrame({"Harmonic propagation": overall_summary,
                           f"k-NN baseline (k={best_k})": baseline_summary})
comparison["Difference"] = comparison.iloc[:, 0] - comparison.iloc[:, 1]
display(comparison.round(3))

results_df.to_csv(os.path.join(OUT_DIR, "harmonic_results_per_fold.csv"), index=False)
baseline_all_df.to_csv(os.path.join(OUT_DIR, "knn_results_per_fold.csv"), index=False)
comparison.to_csv(os.path.join(OUT_DIR, "model_comparison.csv"))

### F_max (threshold-free F-measure)

Precision, recall and F1 above use a fixed cut-off of 0.5 after min-max scaling, which depends on the scaling and differs between harmonic propagation and k-NN. F_max avoids this: the same thresholds 0.01-1.00 are applied to the raw scores of both methods (both lie between 0 and 1), and the best F-measure is reported. Table 1 of the manuscript (AUROC, AUPRC, F_max) is written to `table1_auroc_auprc_fmax.csv`.

In [ ]:
# ---------------- F_max: threshold-free F-measure (replaces min-max scaling + 0.5 cut-off) ----------------
# Harmonic and k-NN scores both lie in [0, 1], so the same thresholds are applied to both without normalisation.
# For each threshold t: a side effect is predicted for a test drug if its score >= t;
#   precision = mean over test drugs with >= 1 predicted side effect,
#   recall    = mean over test drugs with >= 1 true side effect,
#   F(t) = 2PR / (P + R);  F_max = max over t.  Computed per fold, then averaged (Radivojac et al., 2013).
FMAX_THRESHOLDS = np.round(np.arange(0.01, 1.0001, 0.01), 2)


def fmax_curve(Y_true, S, thresholds=FMAX_THRESHOLDS):    # S = score matrix, P = prediction matrix
    rows = []
    n_true = Y_true.sum(axis=1)
    has_true = n_true > 0
    for t in thresholds:
        P = S >= t
        tp = (P & (Y_true == 1)).sum(axis=1)
        n_pred = P.sum(axis=1)
        covered = has_true & (n_pred > 0)
        prec = (tp[covered] / n_pred[covered]).mean() if covered.any() else 0.0
        rec = (tp[has_true] / n_true[has_true]).mean()
        f = 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0
        rows.append({"threshold": t, "precision": prec, "recall": rec, "F": f})
    return pd.DataFrame(rows)


def fmax_per_fold(score_fn):
    """score_fn(train_idx, test_idx) -> (n_test x n_side_effects) score matrix."""
    out = []
    for fold, (train_idx, test_idx) in enumerate(folds, start=1):
        S = score_fn(train_idx, test_idx)
        if TIE_DECIMALS is not None:
            S = np.round(S, TIE_DECIMALS)
        c = fmax_curve(Y_all[test_idx], S)
        best = c.loc[c["F"].idxmax()]
        out.append({"fold": fold, "Fmax": best["F"], "threshold": best["threshold"],
                    "precision": best["precision"], "recall": best["recall"], "curve": c})
    return out


fmax_harmonic = fmax_per_fold(lambda tr, te: oof_scores[te])
fmax_knn = fmax_per_fold(lambda tr, te: knn_scores(W, Y_all, tr, te, best_k))


def threshold_from_other_folds(res):
    """F of each fold at the threshold that is best on the other four folds (no tuning on the test fold)."""
    vals = []
    for i, r in enumerate(res):
        others = pd.concat([res[j]["curve"] for j in range(len(res)) if j != i]).groupby("threshold")["F"].mean()
        vals.append(r["curve"].set_index("threshold").loc[others.idxmax(), "F"])
    return np.mean(vals)


fmax_table = pd.DataFrame({
    "Harmonic propagation": [r["Fmax"] for r in fmax_harmonic],
    f"k-NN baseline (k={best_k})": [r["Fmax"] for r in fmax_knn],
    "threshold harmonic": [r["threshold"] for r in fmax_harmonic],
    "threshold k-NN": [r["threshold"] for r in fmax_knn],
}, index=pd.Index(range(1, len(folds) + 1), name="fold"))
display(fmax_table.round(3))
print("F with the threshold chosen on the other folds: harmonic %.3f, k-NN %.3f"
      % (threshold_from_other_folds(fmax_harmonic), threshold_from_other_folds(fmax_knn)))

# Table 1 of the manuscript: AUROC, AUPRC, F_max
table1 = pd.DataFrame({
    "Harmonic propagation": [overall_summary["AUC_ROC"], overall_summary["AUPRC"], fmax_table.iloc[:, 0].mean()],
    f"k-NN baseline (k={best_k})": [baseline_summary["AUC_ROC"], baseline_summary["AUPRC"], fmax_table.iloc[:, 1].mean()],
}, index=["AUROC", "AUPRC", "Fmax"])
table1["Difference"] = table1.iloc[:, 0] - table1.iloc[:, 1]
display(table1.round(3))
table1.to_csv(os.path.join(OUT_DIR, "table1_auroc_auprc_fmax.csv"))
fmax_table.to_csv(os.path.join(OUT_DIR, "fmax_per_fold.csv"))

n_per_adr = Y_all.sum(axis=0)
common = n_per_adr >= 0.1 * n_drugs
print(f"{common.sum()} side effects reported for >= 10% of drugs hold "
      f"{Y_all[:, common].sum() / Y_all.sum():.0%} of all drug-side effect associations")

# F_max of the shuffled-network control (same permutation as in the control cell)
perm_f = np.random.default_rng(SEED).permutation(n_drugs)
W_perm = W[np.ix_(perm_f, perm_f)]


def harmonic_scores_perm(tr, te):
    A, B = harmonic_system(W_perm, tr, te)
    return np.linalg.solve(A, B @ Y_all[tr])


fmax_control = np.mean([r["Fmax"] for r in fmax_per_fold(harmonic_scores_perm)])
print(f"F_max with drug labels shuffled: {fmax_control:.3f}")

## 6. Predictability of individual ADRs

Mean AUPRC is not comparable between ADRs: its chance level equals the share of drugs that have the ADR, so very
common ADRs score high and ADRs with two or three drugs score 0 or 1 by chance. ADRs are therefore ranked by
mean AUROC, and only if they have enough drugs on both sides (`MIN_POSITIVE_DRUGS`) and were evaluated in every fold.

For each ranked ADR the table also gives

* `AUROC_pooled` and `p_value`: AUROC of the out-of-fold scores of all drugs and a one-sided Mann-Whitney U test
  (positive drugs score higher than negative drugs), with Benjamini-Hochberg `fdr`;
* `sim_within`: mean gene similarity between pairs of drugs that have the ADR;
* `sim_between`: mean gene similarity between drugs that have the ADR and drugs that do not;
* `target_coherence` = `sim_within` - `sim_between`.

ADRs annotated to exactly the same set of drugs are merged (`same_drugs_as`).

In [ ]:
def benjamini_hochberg(p):
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    adj = p[order] * len(p) / np.arange(1, len(p) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty_like(adj)
    out[order] = np.minimum(adj, 1.0)
    return out


side_effect_summary = (
    results_df.groupby("side_effect")
    .agg(AUROC_mean=("AUC_ROC", "mean"), AUROC_std=("AUC_ROC", "std"),
         AUPRC_mean=("AUPRC", "mean"), AUPRC_std=("AUPRC", "std"),
         Precision_mean=("Precision", "mean"), Recall_mean=("Recall", "mean"),
         F1_mean=("F1_Score", "mean"), prevalence_test_mean=("prevalence_test", "mean"),
         n_folds=("fold", "nunique"))
    .reset_index()
)
side_effect_summary["AUPRC_gain_mean"] = side_effect_summary["AUPRC_mean"] - side_effect_summary["prevalence_test_mean"]

# per-ADR quantities from the label matrix, the gene similarities and the out-of-fold scores
col_of = {name: j for j, name in enumerate(side_effects)}
extra = []
for name in side_effect_summary["side_effect"]:
    j = col_of[name]
    pos = Y_all[:, j] == 1
    k = int(pos.sum())
    within = W_full[np.ix_(pos, pos)].sum() / (k * (k - 1)) if k > 1 else np.nan      # diagonal of W_full is 0
    between = W_full[np.ix_(pos, ~pos)].mean()
    u = mannwhitneyu(oof_scores[pos, j], oof_scores[~pos, j], alternative="greater")
    extra.append({"n_positive_drugs": k, "n_negative_drugs": n_drugs - k,
                  "AUROC_pooled": u.statistic / (k * (n_drugs - k)), "p_value": u.pvalue,
                  "sim_within": within, "sim_between": between, "target_coherence": within - between})
side_effect_summary = pd.concat([side_effect_summary, pd.DataFrame(extra)], axis=1)

# merge ADRs that have an identical drug profile
side_effect_summary["profile"] = [Y_all[:, col_of[name]].tobytes() for name in side_effect_summary["side_effect"]]
side_effect_summary = side_effect_summary.sort_values("side_effect").reset_index(drop=True)
aliases = side_effect_summary.groupby("profile")["side_effect"].apply(list)
unique_adrs = side_effect_summary.drop_duplicates("profile").copy()
unique_adrs["same_drugs_as"] = unique_adrs["profile"].map(lambda p: "; ".join(aliases[p][1:]))

# ADRs that can be ranked
eligible = (unique_adrs["n_positive_drugs"] >= MIN_POSITIVE_DRUGS) & (unique_adrs["n_negative_drugs"] >= MIN_POSITIVE_DRUGS)
if REQUIRE_ALL_FOLDS:
    eligible &= unique_adrs["n_folds"] == N_SPLITS
paper_table = unique_adrs[eligible].copy()
paper_table["fdr"] = benjamini_hochberg(paper_table["p_value"])
paper_table["significant"] = paper_table["fdr"] < FDR_ALPHA

sort_keys = list(dict.fromkeys([RANK_BY, "AUROC_mean", "AUPRC_gain_mean", "n_positive_drugs"]))
paper_table = (paper_table.sort_values(sort_keys + ["side_effect"], ascending=[False] * len(sort_keys) + [True])
               .drop(columns="profile").reset_index(drop=True))
paper_table.insert(0, "Rank", paper_table.index + 1)
side_effect_summary = side_effect_summary.drop(columns="profile")
paper_table.to_csv(os.path.join(OUT_DIR, "ranked_side_effects.csv"), index=False)

n_tied = int((paper_table[RANK_BY] == paper_table[RANK_BY].max()).sum())
print(f"Side effects evaluated in at least one fold: {len(side_effect_summary)}")
print(f"Distinct drug profiles: {len(unique_adrs)}")
print(f"Ranked (>= {MIN_POSITIVE_DRUGS} drugs with and without the ADR"
      f"{', all folds' if REQUIRE_ALL_FOLDS else ''}): {len(paper_table)}")
print(f"ADRs sharing the highest {RANK_BY}: {n_tied}")
print(f"Mean AUROC of ranked ADRs: {paper_table['AUROC_mean'].mean():.3f} "
      f"(median {paper_table['AUROC_mean'].median():.3f}, "
      f"interquartile range {paper_table['AUROC_mean'].quantile(0.25):.3f}-{paper_table['AUROC_mean'].quantile(0.75):.3f}, "
      f"range {paper_table['AUROC_mean'].min():.3f}-{paper_table['AUROC_mean'].max():.3f})")
print(f"Predicted better than chance at FDR < {FDR_ALPHA}: {int(paper_table['significant'].sum())} "
      f"of {len(paper_table)} ({100 * paper_table['significant'].mean():.0f}%)")

top10_adrs = paper_table.head(N_EXTREME)
bottom10_adrs = paper_table.tail(N_EXTREME)
show_cols = ["Rank", "side_effect", "AUROC_mean", "AUROC_std", "AUROC_pooled", "fdr", "AUPRC_mean",
             "n_positive_drugs", "sim_within", "sim_between", "same_drugs_as"]
print(f"\nTop {N_EXTREME} ADRs")
display(top10_adrs[show_cols].round(3))
print(f"Bottom {N_EXTREME} ADRs")
display(bottom10_adrs[show_cols].round(3))

### What goes with predictability?

Spearman correlations across the ranked ADRs. If mean AUROC is a fair measure it should not depend on how many drugs
have the ADR, while AUPRC does. The correlation with target coherence tests whether ADRs are predicted well when the
drugs that cause them share gene targets.

In [ ]:
def spearman_row(x, y, label):
    rho, p = spearmanr(paper_table[x], paper_table[y])
    return {"comparison": label, "Spearman rho": rho, "p": p}


correlations = pd.DataFrame([
    spearman_row("n_positive_drugs", "AUROC_mean", "number of drugs vs AUROC"),
    spearman_row("n_positive_drugs", "AUPRC_mean", "number of drugs vs AUPRC"),
    spearman_row("sim_within", "AUROC_mean", "similarity within ADR drugs vs AUROC"),
    spearman_row("sim_between", "AUROC_mean", "similarity to other drugs vs AUROC"),
    spearman_row("target_coherence", "AUROC_mean", "target coherence vs AUROC"),
    spearman_row("target_coherence", "AUPRC_gain_mean", "target coherence vs AUPRC gain"),
])
correlations.to_csv(os.path.join(OUT_DIR, "predictability_correlations.csv"), index=False)
display(correlations.round({"Spearman rho": 3}))

quartile = pd.qcut(paper_table["AUROC_mean"], 4, labels=["Q1 (lowest AUROC)", "Q2", "Q3", "Q4 (highest AUROC)"])
by_quartile = paper_table.groupby(quartile, observed=True).agg(
    n_ADRs=("side_effect", "size"), AUROC_median=("AUROC_mean", "median"),
    sim_within_median=("sim_within", "median"), sim_between_median=("sim_between", "median"),
    share_significant=("significant", "mean"))
by_quartile.to_csv(os.path.join(OUT_DIR, "predictability_quartiles.csv"))
display(by_quartile.round(3))

### Negative control and threshold sensitivity

* **Shuffled network:** the drug labels of the similarity matrix are permuted, which keeps the network structure but
  breaks the link between a drug's neighbours and its side effects. Performance should fall to chance.
* **Threshold:** the cross-validation is repeated for other similarity thresholds. The threshold decides which drugs
  are kept, so each row is evaluated on a different set of drugs.

In [ ]:
def quick_cv(W_mat, Y_mat):
    """Mean test AUROC / AUPRC of harmonic propagation over all evaluable side effect x fold combinations."""
    aurocs, auprcs = [], []
    for train_idx, test_idx in KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(np.arange(len(W_mat))):
        A, B = harmonic_system(W_mat, train_idx, test_idx)
        Y_train, Y_test = Y_mat[train_idx], Y_mat[test_idx]
        F_test = np.linalg.solve(A, B @ Y_train)
        if TIE_DECIMALS is not None:
            F_test = np.round(F_test, TIE_DECIMALS)
        valid = (Y_train.sum(axis=0) > 0) & (Y_test.sum(axis=0) > 0) & (Y_test.sum(axis=0) < len(test_idx))
        for j in np.nonzero(valid)[0]:
            aurocs.append(roc_auc_score(Y_test[:, j], F_test[:, j]))
            auprcs.append(average_precision_score(Y_test[:, j], F_test[:, j]))
    return float(np.mean(aurocs)), float(np.mean(auprcs)), len(aurocs)


if RUN_NETWORK_CONTROL:
    perm = np.random.default_rng(SEED).permutation(n_drugs)
    control_auroc, control_auprc, _ = quick_cv(W[np.ix_(perm, perm)], Y_all)
    network_control = pd.DataFrame(
        {"AUROC": [overall_summary["AUC_ROC"], control_auroc], "AUPRC": [overall_summary["AUPRC"], control_auprc]},
        index=["Drug-gene similarity network", "Same network, drug labels shuffled"])
    network_control.to_csv(os.path.join(OUT_DIR, "network_control.csv"))
    display(network_control.round(3))

In [ ]:
if RUN_THRESHOLD_SENSITIVITY:
    rows = []
    for thr in THRESHOLDS:
        keep_t = (S >= thr).any(axis=1)
        W_t = S[np.ix_(keep_t, keep_t)]
        G_t = nx.from_numpy_array((W_t >= thr).astype(int))
        Y_t = drugs_se.loc[shared_drugs[keep_t]].to_numpy(dtype=float)
        auroc_t, auprc_t, n_eval = quick_cv(W_t, Y_t)
        rows.append({"threshold": thr, "drugs": int(keep_t.sum()),
                     "share_of_shared_drugs": keep_t.mean(), "edges": G_t.number_of_edges(),
                     "components": nx.number_connected_components(G_t),
                     "largest_component": len(max(nx.connected_components(G_t), key=len)),
                     "evaluations": n_eval, "AUROC": auroc_t, "AUPRC": auprc_t})
        print(f"threshold {thr}: done")
    threshold_sensitivity = pd.DataFrame(rows)
    threshold_sensitivity.to_csv(os.path.join(OUT_DIR, "threshold_sensitivity.csv"), index=False)
    display(threshold_sensitivity.round(3))

## 7. Target genes of the top and bottom ADRs

For each ADR every gene is tested for over-representation among the drugs that have the ADR, against all analysed
drugs (hypergeometric test, Benjamini-Hochberg correction across the genes tested for that ADR). Genes with
FDR below `FDR_ALPHA` are the ADR's *associated target genes*. This replaces the union of all genes of all drugs,
which grows with the number of drugs and mixes specific targets with incidental ones.

In [ ]:
G_all = (drug_gene_matrix.to_numpy() > 0)
gene_names = drug_gene_matrix.columns.to_numpy()
gene_totals = G_all.sum(axis=0)                       # number of analysed drugs that interact with each gene
background_genes = gene_names[gene_totals > 0].tolist()


def associated_genes(adr):
    """Genes over-represented among the drugs that have the ADR."""
    pos = Y_all[:, col_of[adr]] == 1
    k_pos = int(pos.sum())
    hits = G_all[pos].sum(axis=0)
    tested = hits >= 2                                # a gene must be shared by at least two of the ADR's drugs
    out = pd.DataFrame({"gene": gene_names[tested], "drugs_with_adr": hits[tested],
                        "drugs_total": gene_totals[tested]})
    out["p_value"] = hypergeom.sf(out["drugs_with_adr"] - 1, n_drugs, out["drugs_total"], k_pos)
    out["fdr"] = benjamini_hochberg(out["p_value"]) if len(out) else []
    out["share_of_adr_drugs"] = out["drugs_with_adr"] / k_pos
    return out.sort_values(["fdr", "p_value"]).reset_index(drop=True)


# number of associated genes for every ranked ADR
n_assoc = []
for adr in paper_table["side_effect"]:
    t = associated_genes(adr)
    n_assoc.append(int((t["fdr"] < FDR_ALPHA).sum()))
paper_table["n_associated_genes"] = n_assoc
paper_table.to_csv(os.path.join(OUT_DIR, "ranked_side_effects.csv"), index=False)
top10_adrs = paper_table.head(N_EXTREME)
bottom10_adrs = paper_table.tail(N_EXTREME)

rho, p = spearmanr(paper_table["n_associated_genes"], paper_table["AUROC_mean"])
print(f"Ranked ADRs with at least one associated gene: {int((paper_table['n_associated_genes'] > 0).sum())} of {len(paper_table)}")
print(f"Number of associated genes vs AUROC: Spearman rho = {rho:.3f}, p = {p:.1e}")
quartile = pd.qcut(paper_table["AUROC_mean"], 4, labels=["Q1 (lowest AUROC)", "Q2", "Q3", "Q4 (highest AUROC)"])
display(paper_table.groupby(quartile, observed=True)["n_associated_genes"]
        .agg(share_with_gene=lambda x: (x > 0).mean(), median_genes="median").round(3))

In [ ]:
def group_table(adr_table, group):
    rows, gene_rows = [], []
    for _, r in adr_table.iterrows():
        t = associated_genes(r["side_effect"])
        sig = t[t["fdr"] < FDR_ALPHA]
        gene_rows.append(sig.assign(group=group, side_effect=r["side_effect"]))
        rows.append({
            "group": group, "Rank": r["Rank"], "side_effect": r["side_effect"],
            "n_drugs": r["n_positive_drugs"], "AUROC_mean": r["AUROC_mean"], "AUROC_std": r["AUROC_std"],
            "fdr": r["fdr"], "sim_within": r["sim_within"], "sim_between": r["sim_between"],
            "n_associated_genes": len(sig),
            "top_genes": ", ".join(f"{g} ({a}/{b})" for g, a, b in
                                   zip(sig["gene"].head(5), sig["drugs_with_adr"].head(5), sig["drugs_total"].head(5))),
            "drugs": ", ".join(drugs_for_adr(r["side_effect"])),
        })
    return pd.DataFrame(rows), pd.concat(gene_rows, ignore_index=True)


top_table, top_gene_table = group_table(top10_adrs, "top")
bottom_table, bottom_gene_table = group_table(bottom10_adrs, "bottom")
adr_genes = pd.concat([top_table, bottom_table], ignore_index=True)
adr_gene_details = pd.concat([top_gene_table, bottom_gene_table], ignore_index=True)
adr_genes.to_csv(os.path.join(OUT_DIR, "top_bottom_adr_genes.csv"), index=False)
adr_gene_details.to_csv(os.path.join(OUT_DIR, "top_bottom_adr_gene_details.csv"), index=False)

top_drugs = set().union(*[drugs_for_adr(a) for a in top10_adrs["side_effect"]])
bottom_drugs = set().union(*[drugs_for_adr(a) for a in bottom10_adrs["side_effect"]])
top_genes = sorted(set(top_gene_table["gene"]))
bottom_genes = sorted(set(bottom_gene_table["gene"]))

pd.Series(top_genes).to_csv(os.path.join(OUT_DIR, f"top{N_EXTREME}_genes.txt"), index=False, header=False)
pd.Series(bottom_genes).to_csv(os.path.join(OUT_DIR, f"bottom{N_EXTREME}_genes.txt"), index=False, header=False)
pd.Series(background_genes).to_csv(os.path.join(OUT_DIR, "background_genes.txt"), index=False, header=False)
print(f"Associated genes: top group {len(top_genes)}, bottom group {len(bottom_genes)}; background {len(background_genes)} genes")
print("(gene: drugs with the ADR that interact with it / all analysed drugs that interact with it)")
display(adr_genes.drop(columns="drugs").round(3))

### KEGG and GO enrichment

The gene-set libraries are downloaded from Enrichr and the over-representation test is done here, so the
**background is the set of genes in the drug-gene matrix** and not the whole genome: every gene in this dataset is a
drug target, and against the genome almost any pathway would look enriched. A term is tested if at least
`MIN_TERM_GENES` of its genes are in the background. One enrichment is run for each ADR of the top and bottom groups,
using that ADR's associated target genes, and one for each group as a whole.

`ENRICHR_LIBRARIES` may hold exact library names or just the beginning of a name (for example `KEGG`); the newest
matching library is then used and its name is printed. This cell needs internet access.

In [ ]:
MIN_TERM_GENES = 5

def resolve_library(requested, available):
    """Exact library name if it exists, otherwise the newest human library whose name starts with the request."""
    if requested in available:
        return requested
    import re
    stem = re.sub(r"_?(19|20)\d{2}.*$", "", requested)
    candidates = [n for n in available if n.startswith(stem) and re.search(r"(19|20)\d{2}", n)
                  and "Mouse" not in n and "Fly" not in n and "Yeast" not in n and "Worm" not in n and "Fish" not in n]
    if not candidates:
        raise ValueError(f"No Enrichr library matches '{requested}'")
    return max(candidates, key=lambda n: int(re.search(r"(19|20)\d{2}", n).group()))


def over_representation(genes, library, library_name, label, group):
    """Hypergeometric test of every term of a library against the background genes."""
    background = set(background_genes)
    query = set(genes) & background
    rows = []
    for term, members in library.items():
        members = set(members) & background
        if len(members) < MIN_TERM_GENES:
            continue
        overlap = sorted(query & members)
        rows.append({"Gene_set": library_name, "Term": term, "overlap": len(overlap), "term_genes": len(members),
                     "P-value": hypergeom.sf(len(overlap) - 1, len(background), len(members), len(query)),
                     "Genes": ";".join(overlap)})
    out = pd.DataFrame(rows)
    out["Adjusted P-value"] = benjamini_hochberg(out["P-value"])
    out["group"], out["gene_list"], out["n_input_genes"] = group, label, len(query)
    return out.sort_values("P-value")


enrichment = pd.DataFrame()
if RUN_ENRICHR:
    try:
        try:
            import gseapy as gp
        except ImportError:
            import subprocess, sys
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gseapy"])
            import gseapy as gp

        available = gp.get_library_name(organism="Human")
        libraries = {}
        for requested in ENRICHR_LIBRARIES:
            name = resolve_library(requested, available)
            libraries[name] = gp.get_library(name=name, organism="Human")
            n_tested = sum(len(set(m) & set(background_genes)) >= MIN_TERM_GENES for m in libraries[name].values())
            print(f"Library {name}: {len(libraries[name])} terms, {n_tested} with at least {MIN_TERM_GENES} background genes")

        gene_lists = [("top", "all top ADRs", top_genes), ("bottom", "all bottom ADRs", bottom_genes)]
        for group, table in [("top", top_gene_table), ("bottom", bottom_gene_table)]:
            for adr, sub in table.groupby("side_effect", sort=False):
                gene_lists.append((group, adr, sub["gene"].tolist()))

        parts = [over_representation(genes, lib, name, label, group)
                 for group, label, genes in gene_lists if len(genes) >= 3
                 for name, lib in libraries.items()]
        enrichment = pd.concat(parts, ignore_index=True)
        enrichment.to_csv(os.path.join(OUT_DIR, "enrichr_results.csv"), index=False)

        sig = enrichment[enrichment["Adjusted P-value"] < FDR_ALPHA]
        summary = (enrichment.groupby(["group", "gene_list", "Gene_set", "n_input_genes"], sort=False)
                   .agg(terms_tested=("Term", "size")).reset_index())
        best = (sig.sort_values("Adjusted P-value").groupby(["gene_list", "Gene_set"], sort=False)
                .agg(n_significant_terms=("Term", "size"),
                     top_terms=("Term", lambda x: "; ".join(x.head(3))),
                     best_adjusted_p=("Adjusted P-value", "min")).reset_index())
        summary = summary.merge(best, on=["gene_list", "Gene_set"], how="left")
        summary["n_significant_terms"] = summary["n_significant_terms"].fillna(0).astype(int)
        summary.to_csv(os.path.join(OUT_DIR, "enrichr_summary.csv"), index=False)
        display(summary)
    except Exception as err:
        import traceback
        print("Enrichment step skipped:", repr(err))
        traceback.print_exc()

## 8. Figures

Each figure is saved to `figures/` as PNG (300 dpi) and PDF.
Blue and orange are the only two category colours; they stay distinguishable under colour-vision deficiency.

In [ ]:
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "pdf.fonttype": 42,
})


def save_figure(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"))
    plt.show()

### Figure 1. Do drugs with similar gene profiles have similar side-effect profiles?

In [ ]:
iu = np.triu_indices(n_drugs, k=1)
pair_gene = W_full[iu]
pair_se = se_sim.to_numpy()[iu]
rho, p_value = spearmanr(pair_gene, pair_se)

edges = [0, 1e-9, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]
labels = ["0", "0-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4", "0.4-0.5", "0.5-0.6", "0.6-0.8", "0.8-1"]
bins = pd.cut(pair_gene, bins=edges, labels=labels, include_lowest=True)
binned = pd.DataFrame({"bin": bins, "se": pair_se}).groupby("bin", observed=False)["se"]
q = binned.quantile([0.25, 0.5, 0.75]).unstack()
counts = binned.size()

fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(len(labels))
ax.vlines(x, q[0.25], q[0.75], color=BLUE, linewidth=2, alpha=0.45)
ax.plot(x, q[0.5], "-o", color=BLUE, linewidth=2, markersize=6)
for xi, n in zip(x, counts):
    ax.annotate(f"n={n:,}", (xi, 0), xytext=(0, 4), textcoords="offset points",
                ha="center", fontsize=7, color=MUTED)
ax.axvline(labels.index("0.3-0.4") - 0.5, color=MUTED, linestyle="--", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, None)
ax.set_xlabel("Drug-gene Jaccard similarity of a drug pair")
ax.set_ylabel("Side-effect Jaccard similarity\n(median and interquartile range)")
ax.set_title(f"Side-effect similarity by gene similarity of drug pairs (Spearman rho = {rho:.2f})", loc="left")
ax.grid(axis="x", visible=False)
save_figure(fig, "fig1_gene_vs_side_effect_similarity")
print(f"Spearman rho = {rho:.3f}, p = {p_value:.2e}, drug pairs = {len(pair_gene):,}; dashed line = network threshold")

### Figure 2. Drug similarity network with the training and test drugs of one fold

In [ ]:
FOLD_TO_SHOW = 1
train_idx, test_idx = folds[FOLD_TO_SHOW - 1]
test_nodes = [common_drugs[i] for i in test_idx]
train_nodes = [common_drugs[i] for i in train_idx]

pos = nx.spring_layout(threshold_graph, k=0.25, seed=SEED)

fig, ax = plt.subplots(figsize=(8, 7))
nx.draw_networkx_edges(threshold_graph, pos, ax=ax, edge_color=MUTED, alpha=0.25, width=0.6)
nx.draw_networkx_nodes(threshold_graph, pos, nodelist=train_nodes, ax=ax, node_size=28,
                       node_color=BLUE, edgecolors="white", linewidths=0.5)
nx.draw_networkx_nodes(threshold_graph, pos, nodelist=test_nodes, ax=ax, node_size=28,
                       node_color=ORANGE, node_shape="s", edgecolors="white", linewidths=0.5)
ax.legend(handles=[
    Line2D([0], [0], marker="o", linestyle="", color=BLUE, label=f"Training drugs, labels known (n={len(train_nodes)})"),
    Line2D([0], [0], marker="s", linestyle="", color=ORANGE, label=f"Test drugs, labels predicted (n={len(test_nodes)})"),
], loc="upper left")
ax.set_title(f"Drug similarity network, fold {FOLD_TO_SHOW} (edges: gene similarity >= {SIM_THRESHOLD})", loc="left")
ax.axis("off")
save_figure(fig, "fig2_network_train_test")

### Figure 3. AUROC against AUPRC for every side effect

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
plot_data = unique_adrs.sort_values("n_positive_drugs")                  # frequent ADRs drawn on top
sc = ax.scatter(plot_data["AUROC_mean"], plot_data["AUPRC_mean"], c=plot_data["n_positive_drugs"],
                cmap="Blues", norm=LogNorm(vmin=1, vmax=plot_data["n_positive_drugs"].max()),
                s=14, alpha=0.8, edgecolors=MUTED, linewidths=0.2)
ax.axvline(0.5, color=MUTED, linestyle="--", linewidth=1)
ax.text(0.5, 1.02, "chance AUROC", ha="center", va="bottom", fontsize=8, color=MUTED)
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Mean AUROC across folds")
ax.set_ylabel("Mean AUPRC across folds")
ax.set_title("AUROC and AUPRC of each side effect", loc="left", pad=16)
fig.colorbar(sc, ax=ax, label="Number of drugs with the side effect (log scale)")
save_figure(fig, "fig3_auroc_vs_auprc")

### Figure 4. Mean ROC and precision-recall curves

In [ ]:
mean_prevalence = (results_df["n_pos_test"] / (results_df[["TP", "TN", "FP", "FN"]].sum(axis=1))).mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for fold, curves in fold_curves.items():
    axes[0].plot(roc_grid, curves["tpr"], color=BLUE, alpha=0.35, linewidth=1)
    axes[1].plot(pr_grid, curves["prec"], color=BLUE, alpha=0.35, linewidth=1)
axes[0].plot(roc_grid, np.mean([c["tpr"] for c in fold_curves.values()], axis=0), color=BLUE, linewidth=2.2)
axes[1].plot(pr_grid, np.mean([c["prec"] for c in fold_curves.values()], axis=0), color=BLUE, linewidth=2.2)

axes[0].plot([0, 1], [0, 1], "--", color=MUTED, linewidth=1)
axes[1].axhline(mean_prevalence, linestyle="--", color=MUTED, linewidth=1)

axes[0].set(xlabel="False positive rate", ylabel="True positive rate", xlim=(0, 1), ylim=(0, 1.01))
axes[0].set_title(f"a  ROC (mean AUROC = {overall_summary['AUC_ROC']:.3f})", loc="left")
axes[1].set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, None))
axes[1].set_title(f"b  Precision-recall (mean AUPRC = {overall_summary['AUPRC']:.3f})", loc="left")
fig.legend(handles=[
    Line2D([0], [0], color=BLUE, linewidth=2.2, label="Mean of the five folds"),
    Line2D([0], [0], color=BLUE, alpha=0.35, linewidth=1, label="Individual folds"),
    Line2D([0], [0], color=MUTED, linestyle="--", linewidth=1, label="Chance level"),
], loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.06))
fig.tight_layout()
save_figure(fig, "fig4_roc_pr_curves")

### Figure 5. AUPRC and AUROC against the number of drugs that have the side effect

In [ ]:
def binned_median(ax, data, x, y, colour):
    edges = [e for e in [2, 3, 5, 10, 20, 50, 100, 200, 400] if e <= data[x].max()] + [data[x].max() + 1]
    groups = data.groupby(pd.cut(data[x], bins=edges, right=False), observed=True)
    q = groups[y].quantile([0.25, 0.5, 0.75]).unstack()
    centres = groups[x].median()
    ax.fill_between(centres, q[0.25], q[0.75], color=colour, alpha=0.2, linewidth=0)
    ax.plot(centres, q[0.5], "-o", color=colour, linewidth=2, markersize=5)


plot_data = unique_adrs
jitter = np.random.default_rng(SEED).uniform(0.94, 1.06, len(plot_data))      # separates identical integer counts
ticks = [t for t in [2, 5, 10, 20, 50, 100, 200, 400] if t <= plot_data["n_positive_drugs"].max()]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True)
for ax, y, label in zip(axes, ["AUPRC_mean", "AUROC_mean"], ["a  Mean AUPRC", "b  Mean AUROC"]):
    ax.scatter(plot_data["n_positive_drugs"] * jitter, plot_data[y], s=7, color=MUTED, alpha=0.3, linewidths=0)
    binned_median(ax, plot_data, "n_positive_drugs", y, BLUE)
    ax.axvline(MIN_POSITIVE_DRUGS, color=MUTED, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(t) for t in ticks])
    ax.minorticks_off()
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Number of drugs with the side effect (log scale)")
    ax.set_ylabel(label[3:] + " across folds")
    ax.set_title(label, loc="left")
axes[1].axhline(0.5, color=MUTED, linestyle=":", linewidth=1)
fig.legend(handles=[
    Line2D([0], [0], marker="o", linestyle="", color=MUTED, alpha=0.6, markersize=4, label="One side effect"),
    Line2D([0], [0], marker="o", color=BLUE, linewidth=2, markersize=5, label="Median (band: interquartile range)"),
    Line2D([0], [0], color=MUTED, linestyle="--", linewidth=1, label=f"Minimum for ranking ({MIN_POSITIVE_DRUGS} drugs)"),
], loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.08))
fig.tight_layout()
save_figure(fig, "fig5_metrics_vs_frequency")

### Figure 6. Harmonic propagation against the k-NN baseline

In [ ]:
harmonic_folds = results_df.groupby("fold")[METRICS].mean()
knn_folds = baseline_results_df.groupby("fold")[METRICS].mean()
metric_labels = ["AUROC", "AUPRC", "Precision", "Recall", "F1-score", "Specificity", "Accuracy"]

fig, ax = plt.subplots(figsize=(7, 4))
y = np.arange(len(METRICS))[::-1]
for yi, metric in zip(y, METRICS):
    h, b = harmonic_folds[metric], knn_folds[metric]
    ax.plot([b.mean(), h.mean()], [yi, yi], color=GRID, linewidth=2.5, zorder=1)
    ax.scatter(b, np.full(len(b), yi - 0.12), s=10, color=ORANGE, alpha=0.5, linewidths=0, zorder=2)
    ax.scatter(h, np.full(len(h), yi + 0.12), s=10, color=BLUE, alpha=0.5, linewidths=0, zorder=2)
    ax.scatter(b.mean(), yi, s=70, color=ORANGE, marker="o", edgecolors="white", linewidths=1, zorder=3)
    ax.scatter(h.mean(), yi, s=70, color=BLUE, edgecolors="white", linewidths=1, zorder=3)
ax.set_yticks(y)
ax.set_yticklabels(metric_labels, color=INK)
ax.set_xlim(0, 1)
ax.set_xlabel("Score (large marker: mean of five folds; small markers: individual folds)")
ax.set_title("Harmonic propagation and k-NN baseline, test-set performance", loc="left")
ax.grid(axis="y", visible=False)
ax.legend(handles=[
    Line2D([0], [0], marker="o", linestyle="", color=BLUE, markersize=8, label="Harmonic propagation"),
    Line2D([0], [0], marker="o", linestyle="", color=ORANGE, markersize=8, label=f"k-NN baseline (k={best_k})"),
], loc="upper center", ncol=2, bbox_to_anchor=(0.5, -0.2))
save_figure(fig, "fig6_harmonic_vs_knn")

### Figure 7. Predictability against target coherence

In [ ]:
rho, p = spearmanr(paper_table["target_coherence"], paper_table["AUROC_mean"])
middle = paper_table.iloc[N_EXTREME:-N_EXTREME]

fig, ax = plt.subplots(figsize=(6.5, 4.8))
ax.scatter(middle["target_coherence"], middle["AUROC_mean"], s=12, color=MUTED, alpha=0.4, linewidths=0)
ax.scatter(top10_adrs["target_coherence"], top10_adrs["AUROC_mean"], s=42, color=BLUE,
           edgecolors="white", linewidths=0.8, zorder=3)
ax.scatter(bottom10_adrs["target_coherence"], bottom10_adrs["AUROC_mean"], s=42, color=ORANGE,
           edgecolors="white", linewidths=0.8, zorder=3)
ax.axhline(0.5, color=MUTED, linestyle=":", linewidth=1)
ax.axvline(0, color=MUTED, linestyle=":", linewidth=1)
ax.set_xlabel("Target coherence\n(gene similarity among the ADR's drugs minus similarity to other drugs)")
ax.set_ylabel("Mean AUROC across folds")
ax.set_title(f"Predictability and target coherence of {len(paper_table)} ADRs (Spearman rho = {rho:.2f})", loc="left")
ax.legend(handles=[
    Line2D([0], [0], marker="o", linestyle="", color=BLUE, markersize=7, label=f"{N_EXTREME} highest AUROC"),
    Line2D([0], [0], marker="o", linestyle="", color=ORANGE, markersize=7, label=f"{N_EXTREME} lowest AUROC"),
    Line2D([0], [0], marker="o", linestyle="", color=MUTED, alpha=0.6, markersize=4, label="Other ranked ADRs"),
], loc="lower right")
save_figure(fig, "fig7_auroc_vs_target_coherence")
print(f"Spearman rho = {rho:.3f}, p = {p:.1e}")

### Figure 8. The most and least predictable ADRs

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 6.4), sharex=True)
for ax, table, colour, marker, title in [
        (axes[0], top10_adrs, BLUE, "o", f"a  {N_EXTREME} highest mean AUROC"),
        (axes[1], bottom10_adrs.iloc[::-1], ORANGE, "o", f"b  {N_EXTREME} lowest mean AUROC")]:
    y = np.arange(len(table))[::-1]
    ax.hlines(y, table["AUROC_mean"] - table["AUROC_std"], table["AUROC_mean"] + table["AUROC_std"],
              color=colour, alpha=0.45, linewidth=2)
    ax.scatter(table["AUROC_mean"], y, s=45, color=colour, marker=marker, edgecolors="white", linewidths=0.8, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{name.capitalize()} (n={n})" for name, n in zip(table["side_effect"], table["n_positive_drugs"])],
                       color=INK)
    ax.axvline(0.5, color=MUTED, linestyle="--", linewidth=1)
    ax.set_title(title, loc="left")
    ax.grid(axis="y", visible=False)
axes[1].set_xlim(0, 1.02)
#axes[1].set_xlabel("Mean AUROC across five folds (bar: standard deviation; n: drugs with the ADR; dashed: chance)")
fig.tight_layout()
save_figure(fig, "fig8_top_bottom_adrs")

### Supplementary figure. Consistency across folds

In [ ]:
fig, axes = plt.subplots(1, N_SPLITS, figsize=(3 * N_SPLITS, 3.2), sharex=True, sharey=True)
for ax, fold in zip(axes, range(1, N_SPLITS + 1)):
    d = results_df[results_df["fold"] == fold]
    ax.scatter(d["AUC_ROC"], d["AUPRC"], s=5, color=BLUE, alpha=0.25, linewidths=0)
    ax.set_title(f"Fold {fold}  (n={len(d):,})", loc="left")
    ax.set_xlabel("AUROC")
    ax.set(xlim=(0, 1.02), ylim=(0, 1.02))
axes[0].set_ylabel("AUPRC")
fig.suptitle("Each point is one side effect evaluated on the test drugs of that fold", x=0.01, ha="left", fontsize=10)
fig.tight_layout()
save_figure(fig, "figS1_auroc_vs_auprc_per_fold")

## 9. Enrichment across the whole ranking

The enrichment is repeated for every ranked ADR that has at least three associated target genes, and the results are
compared between the best and worst predicted ADRs at three group sizes (10, 100 and half of the ranking).

* `enrichment_group_comparison.csv`: share of ADRs that can be tested and that have at least one significant term.
* `pathway_predictability.csv`: for each pathway, how many top and bottom ADRs are enriched for it (Fisher's exact
  test), and whether the ADRs enriched for it have a different AUROC from the rest (Mann-Whitney U test, no cut-off).
* `enrichment_per_adr.csv`, `enrichment_significant_terms_all_adrs.csv`: the underlying results.

ADRs with too few associated genes count as "no significant term". Needs the libraries loaded by the enrichment cell.

In [ ]:
# ---- Enrichment across the whole ranking: top/bottom 10, 100 and half, and per pathway ----
from scipy.stats import fisher_exact

GROUP_SIZES = [10, 100, len(paper_table) // 2]     # top k against bottom k
MIN_GENES_FOR_ENRICHMENT = 3                       # an ADR needs this many associated genes to be tested
MIN_ADRS_PER_TERM = 5                              # a pathway is compared between groups if enriched in this many ADRs

if "libraries" not in globals() or len(libraries) == 0:
    print("Run the enrichment cell first (the gene-set libraries are not loaded).")
else:
    bg_index = {g: i for i, g in enumerate(background_genes)}
    n_bg = len(background_genes)

    # associated genes of every ranked ADR, as a 0/1 matrix (ADRs x background genes)
    adr_names = paper_table["side_effect"].tolist()
    Q = np.zeros((len(adr_names), n_bg), dtype=int)
    for i, adr in enumerate(adr_names):
        t = associated_genes(adr)
        for g in t.loc[t["fdr"] < FDR_ALPHA, "gene"]:
            Q[i, bg_index[g]] = 1
    n_query = Q.sum(axis=1)
    testable = n_query >= MIN_GENES_FOR_ENRICHMENT

    per_adr = paper_table[["Rank", "side_effect", "AUROC_mean", "n_positive_drugs"]].copy()
    per_adr["n_associated_genes"] = n_query
    per_adr["testable"] = testable
    significant_terms, pathway_tables = [], []

    for lib_name, lib in libraries.items():
        # gene sets of the library, as a 0/1 matrix (terms x background genes)
        terms, rows = [], []
        for term, members in lib.items():
            idx = [bg_index[g] for g in set(members) if g in bg_index]
            if len(idx) >= MIN_TERM_GENES:
                row = np.zeros(n_bg, dtype=int)
                row[idx] = 1
                terms.append(term)
                rows.append(row)
        M = np.array(rows)
        term_sizes = M.sum(axis=1)

        overlap = Q @ M.T                                                    # ADRs x terms
        pvals = hypergeom.sf(overlap - 1, n_bg, term_sizes[None, :], n_query[:, None])
        adj = np.ones_like(pvals)
        for i in np.nonzero(testable)[0]:
            adj[i] = benjamini_hochberg(pvals[i])
        sig = (adj < FDR_ALPHA) & testable[:, None]

        per_adr[f"n_significant_{lib_name}"] = sig.sum(axis=1)
        ii, jj = np.nonzero(sig)
        significant_terms.append(pd.DataFrame({
            "Rank": per_adr["Rank"].to_numpy()[ii], "side_effect": np.array(adr_names)[ii], "Gene_set": lib_name,
            "Term": np.array(terms)[jj], "overlap": overlap[ii, jj], "term_genes": term_sizes[jj],
            "Adjusted P-value": adj[ii, jj]}))

        # which pathways go with predictable ADRs?
        auroc = per_adr["AUROC_mean"].to_numpy()
        rows = []
        for j in np.nonzero(sig.sum(axis=0) >= MIN_ADRS_PER_TERM)[0]:
            has = sig[:, j]
            row = {"Gene_set": lib_name, "Term": terms[j], "n_ADRs_enriched": int(has.sum()),
                   "median_AUROC_enriched": np.median(auroc[has]), "median_AUROC_other": np.median(auroc[~has]),
                   "p_AUROC": mannwhitneyu(auroc[has], auroc[~has], alternative="two-sided").pvalue}
            for k in GROUP_SIZES:                                             # paper_table is sorted, best first
                top_k, bottom_k = int(has[:k].sum()), int(has[-k:].sum())
                row[f"top{k}"], row[f"bottom{k}"] = top_k, bottom_k
                row[f"p_top{k}_vs_bottom{k}"] = fisher_exact([[top_k, k - top_k], [bottom_k, k - bottom_k]])[1]
            rows.append(row)
        table = pd.DataFrame(rows)
        if len(table):
            for col in [c for c in table.columns if c.startswith("p_")]:
                table[col.replace("p_", "fdr_", 1)] = benjamini_hochberg(table[col])
            pathway_tables.append(table.sort_values("p_AUROC"))
        print(f"{lib_name}: {len(terms)} terms tested, {int(sig.any(axis=1).sum())} ADRs with at least one significant term, "
              f"{len(table)} terms enriched in at least {MIN_ADRS_PER_TERM} ADRs")

    significant_terms = pd.concat(significant_terms, ignore_index=True)
    pathway_predictability = pd.concat(pathway_tables, ignore_index=True) if pathway_tables else pd.DataFrame()
    per_adr.to_csv(os.path.join(OUT_DIR, "enrichment_per_adr.csv"), index=False)
    significant_terms.to_csv(os.path.join(OUT_DIR, "enrichment_significant_terms_all_adrs.csv"), index=False)
    pathway_predictability.to_csv(os.path.join(OUT_DIR, "pathway_predictability.csv"), index=False)

    # top k against bottom k
    rows = []
    for k in GROUP_SIZES:
        for group, part in [("top", per_adr.head(k)), ("bottom", per_adr.tail(k))]:
            row = {"k": k, "group": group, "median_AUROC": part["AUROC_mean"].median(),
                   "median_associated_genes": part["n_associated_genes"].median(),
                   "share_testable": part["testable"].mean()}
            for lib_name in libraries:
                n_sig = part[f"n_significant_{lib_name}"]
                row[f"share_with_term_{lib_name}"] = (n_sig > 0).mean()
                row[f"median_terms_{lib_name}"] = n_sig[part["testable"]].median() if part["testable"].any() else np.nan
            rows.append(row)
    group_comparison = pd.DataFrame(rows)
    for lib_name in libraries:                                                # is the difference in shares significant?
        col = f"n_significant_{lib_name}"
        pv = {k: fisher_exact([[int((per_adr.head(k)[col] > 0).sum()), int((per_adr.head(k)[col] == 0).sum())],
                               [int((per_adr.tail(k)[col] > 0).sum()), int((per_adr.tail(k)[col] == 0).sum())]])[1]
              for k in GROUP_SIZES}
        group_comparison[f"p_share_{lib_name}"] = group_comparison["k"].map(pv)
    group_comparison.to_csv(os.path.join(OUT_DIR, "enrichment_group_comparison.csv"), index=False)
    print("\nTop k against bottom k (shares are of all ADRs in the group; medians of terms are among testable ADRs)")
    display(group_comparison.round(3))

    for lib_name in libraries:
        rho, p = spearmanr(per_adr.loc[per_adr["testable"], f"n_significant_{lib_name}"],
                           per_adr.loc[per_adr["testable"], "AUROC_mean"])
        print(f"{lib_name}: number of significant terms vs AUROC among testable ADRs, Spearman rho = {rho:.3f} (p = {p:.1e})")

    if len(pathway_predictability):
        k_mid = GROUP_SIZES[1] if len(GROUP_SIZES) > 1 else GROUP_SIZES[0]
        show = ["Gene_set", "Term", "n_ADRs_enriched", "median_AUROC_enriched", "median_AUROC_other", "fdr_AUROC",
                f"top{k_mid}", f"bottom{k_mid}", f"fdr_top{k_mid}_vs_bottom{k_mid}"]
        print("\nPathways whose ADRs differ most in AUROC from the rest (full table in pathway_predictability.csv)")
        display(pathway_predictability.sort_values("p_AUROC").groupby("Gene_set", sort=False).head(12)[show].round(4))

    # figure: enrichment along the ranking
    first_lib = list(libraries)[0]
    decile = pd.qcut(per_adr["Rank"], 10, labels=False)                       # 0 = best predicted tenth
    by_decile = per_adr.groupby(decile).agg(
        testable=("testable", "mean"), with_term=(f"n_significant_{first_lib}", lambda x: (x > 0).mean()),
        auroc=("AUROC_mean", "median"))
    fig, ax = plt.subplots(figsize=(6.5, 4))
    x = np.arange(1, 11)
    ax.plot(x, by_decile["testable"], "-o", color=MUTED, linewidth=2, markersize=6,
            label=f"At least {MIN_GENES_FOR_ENRICHMENT} associated genes")
    ax.plot(x, by_decile["with_term"], "-o", color=BLUE, linewidth=2, markersize=6,
            label=f"At least one significant {first_lib.replace('_', ' ')} pathway")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{i}\n({a:.2f})" for i, a in zip(x, by_decile["auroc"])])
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Tenth of the ADR ranking, best predicted first (median AUROC)")
    ax.set_ylabel("Share of ADRs")
    ax.set_title("Pathway enrichment along the predictability ranking", loc="left")
    ax.legend(loc="lower left")
    ax.grid(axis="x", visible=False)
    save_figure(fig, "fig9_enrichment_along_ranking")

## 10. Download the outputs (Colab)

In [ ]:
# import shutil
# shutil.make_archive("adr_predictability_outputs", "zip", root_dir=".", base_dir=OUT_DIR)
# shutil.make_archive("adr_predictability_figures", "zip", root_dir=".", base_dir=FIG_DIR)
# try:
#     from google.colab import files
#     files.download("adr_predictability_outputs.zip")
#     files.download("adr_predictability_figures.zip")
# except ImportError:
#     print("Archives written to the working directory.")